# 4.6.3 Autoencoder Applications

Anomaly detection via reconstruction error, dimensionality reduction, denoising.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
relu=lambda x:np.maximum(0,x)

h=fetch_california_housing(); X=StandardScaler().fit_transform(h.data)
thr=np.percentile(h.target,75)
X_n=X[h.target<thr]; X_a=X[h.target>=thr]
print('Normal:',len(X_n),'Anomaly:',len(X_a))

In [ ]:
# Train simple AE on normal only, measure reconstruction error
rng=np.random.default_rng(42); nin,nh,nz=8,32,4
We1=rng.standard_normal((nin,nh))*np.sqrt(2/nin); be1=np.zeros(nh)
We2=rng.standard_normal((nh,nz))*np.sqrt(2/nh); be2=np.zeros(nz)
Wd1=rng.standard_normal((nz,nh))*np.sqrt(2/nz); bd1=np.zeros(nh)
Wd2=rng.standard_normal((nh,nin))*np.sqrt(2/nh); bd2=np.zeros(nin)
# Quick training (100 epochs)
for _ in range(100):
    h1=relu(X_n@We1+be1); z=relu(h1@We2+be2); h2=relu(z@Wd1+bd1); xh=h2@Wd2+bd2
    n=len(X_n); dout=2*(xh-X_n)/n; Wd2-=.005*(h2.T@dout); bd2-=.005*dout.sum(0)
    dh2=(dout@Wd2.T)*(h2>0); Wd1-=.005*(z.T@dh2); bd1-=.005*dh2.sum(0)
    dz=(dh2@Wd1.T)*(z>0); We2-=.005*(h1.T@dz); be2-=.005*dz.sum(0)
    dh1=(dz@We2.T)*(h1>0); We1-=.005*(X_n.T@dh1); be1-=.005*dh1.sum(0)
rec=lambda X: (relu(relu(X@We1+be1)@We2+be2)@Wd1+bd1)@Wd2+bd2
err_n=np.mean((rec(X_n)-X_n)**2,1); err_a=np.mean((rec(X_a)-X_a)**2,1)
m=min(len(err_n),len(err_a)); auc=roc_auc_score(np.r_[np.zeros(m),np.ones(m)],np.r_[err_n[:m],err_a[:m]])
print(f'Normal err={err_n.mean():.4f}  Anomaly err={err_a.mean():.4f}  AUC={auc:.4f}')